# 03b — Ranking & Contribution Analysis (Q7–Q11)

> **Session 3B: Ranking window functions and Pareto analysis**

---

## Setup

In [2]:
# ── Connect to database ─────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from shared_setup import get_connection
conn = get_connection()

Connected to existing database: /home/mohamed/Downloads/ITI_Analytical_SQL_project/ecommerce.db
  Fact_Order_Line: 15,000 rows


---
## Q7 — Product Ranking by Category

Rank products within each category by revenue using RANK()

In [28]:
# Q7 — Product Ranking by Category
# TODO: Session 3B
product_rank=conn.execute ( '''
with Category_rev as (

select 

sum(f.gross_amount) as rev_per_category,
D.category_name as cat_name,
f.category_key,
p.product_key,
p.product_name as p_name
from fact_order_line f join Dim_category D
on f.category_key=D.category_key
join Dim_product p on p.product_key= f.product_key
Group by f.category_key,cat_name,p.product_key,p_name
)
select 
cat_name,
category_key,
product_key,
rev_per_category,
p_name,
Rank() over(partition by category_key order by rev_per_category desc) as rnk
from category_rev



''').fetchdf()

print(product_rank.head(20))


   cat_name  category_key  product_key  rev_per_category              p_name  \
0      Home             3           28         272980.07        Air Fryer XL   
1      Home             3           38         250667.11           Knife Set   
2      Home             3           30         244598.70        Mirror Round   
3      Home             3           27         239228.79     Ergonomic Chair   
4      Home             3           32         237904.49       Standing Desk   
5      Home             3           33         228545.69         Blender Max   
6      Home             3           34         221943.69       Pendant Light   
7      Home             3           35         221074.20        Wall Art Set   
8      Home             3           31         217051.50  Memory Foam Pillow   
9      Home             3           36         212663.58    Cotton Sheet Set   
10     Home             3           37         212607.91       Bookshelf Oak   
11     Home             3           39  

---
## Q8 — Product Contribution to Category Revenue

Each product's share of its category total (RATIO_TO_REPORT)

In [39]:
ratio_to_report=conn.execute ( '''
with Category_rev as (

select 

sum(f.gross_amount) as rev_per_category,
D.category_name as cat_name,
f.category_key,
p.product_key,
p.product_name as p_name
from fact_order_line f join Dim_category D
on f.category_key=D.category_key
join Dim_product p on p.product_key= f.product_key
Group by f.category_key,cat_name,p.product_key,p_name
),
total_cat_rev as(
select 
rev_per_category,
cat_name,
category_key,
product_key,
sum(rev_per_category) over(partition by category_key) as rev_per_cat,
p_name
from category_rev
)
 
select 
cat_name,
rev_per_category,
product_key,
rev_per_cat,
p_name,
rev_per_category/rev_per_cat *100 as ratio_to_report
from total_cat_rev


''').fetchdf()

print(ratio_to_report.head(40))


       cat_name  rev_per_category  product_key  rev_per_cat  \
0        Beauty         281568.29           46   3128865.60   
1        Beauty         269060.86           51   3128865.60   
2        Beauty         289287.56           49   3128865.60   
3        Beauty         192454.06           48   3128865.60   
4        Beauty         215654.46           52   3128865.60   
5        Beauty         276896.84           47   3128865.60   
6        Beauty         233700.73           42   3128865.60   
7        Beauty         225129.41           41   3128865.60   
8        Beauty         189598.81           50   3128865.60   
9        Beauty         236405.23           40   3128865.60   
10       Beauty         231775.67           43   3128865.60   
11       Beauty         249034.57           45   3128865.60   
12       Beauty         238299.11           44   3128865.60   
13         Food         245971.68           87   2706116.66   
14         Food         239241.31           84   270611

---
## Q9 — Pareto Analysis (80/20 Rule)

Identify products contributing to 80% of cumulative revenue

In [ ]:
# Q9 — Pareto Analysis (80/20 Rule)
# TODO: Session 3B

pareto_products = conn.execute('''
WITH product_rev AS (
  SELECT
    p.product_key,
    p.product_name,
    SUM(f.gross_amount) AS product_revenue
  FROM Fact_Order_Line f
  JOIN Dim_Product p
    ON f.product_key = p.product_key
  GROUP BY 1, 2
),
ordered AS (
  SELECT
    product_key,
    product_name,
    product_revenue,
    ROW_NUMBER() OVER (ORDER BY product_revenue DESC, product_key) AS rn,
    SUM(product_revenue) OVER (
      ORDER BY product_revenue DESC, product_key
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_revenue
  FROM product_rev
),
total AS (
  SELECT MAX(cumulative_revenue) AS total_revenue
  FROM ordered
),
scored AS (
  SELECT
    o.*,
    (o.cumulative_revenue / NULLIF(t.total_revenue, 0)) * 100 AS cumulative_pct
  FROM ordered o
  CROSS JOIN total t
),
cutoff AS (
  -- first product position where we reach >= 80%
  SELECT MIN(rn) AS cutoff_rn
  FROM scored
  WHERE cumulative_pct >= 80
)
SELECT
  s.product_key,
  s.product_name,
  s.product_revenue,
  s.cumulative_revenue,
  s.cumulative_pct
FROM scored s
CROSS JOIN cutoff c
WHERE s.rn <= c.cutoff_rn
ORDER BY s.rn;
''').fetchdf()

print(pareto_products.head(50))


---
## Q10 — Region Profitability Ranking

Rank regions by profit margin using DENSE_RANK()

In [ ]:
# Q10 — Region Profitability Ranking
# TODO: Session 3B

---
## Q11 — Brand Ranking Within Category

Top brands per category by total profit

In [ ]:
# Q11 — Brand Ranking Within Category
# TODO: Session 3B